In [ ]:
!pip install --upgrade pip
!pip install pandas==3.0.2


In [ ]:
import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")

# Генерируем датасет из 2000 клиентов банка
np.random.seed(42)
N = 2000
names = ['Иван', 'Ольга', 'Дмитрий', 'Екатерина', 'Сергей',
         'Анна', 'Алексей', 'Мария', 'Николай', 'Татьяна']
cities = ['Москва', 'Санкт-Петербург', 'Казань', 'Новосибирск', 'Екатеринбург']

df = pd.DataFrame({
    'client_id': range(1001, 1001 + N),
    'name': np.random.choice(names, N),
    'city': np.random.choice(cities, N),
    'age': np.random.randint(21, 66, N),
    'income': np.random.randint(15000, 500001, N),
    'dti': np.round(np.random.uniform(0, 1, N), 2),
    'default': np.random.choice([True, False], N, p=[0.15, 0.85]),
    'application_date': pd.date_range('2026-01-01', periods=N, freq='h')
})

print(f"Датасет: {df.shape[0]} строк, {df.shape[1]} колонок")
df.head()

Pandas version: 3.0.2
Датасет: 2000 строк, 8 колонок


,client_id,name,city,age,income,dti,default,application_date
0,1001,Алексей,Санкт-Петербург,57,83968,0.31,False,2026-01-01 00:00:00
1,1002,Екатерина,Екатеринбург,35,75430,0.96,False,2026-01-01 01:00:00
2,1003,Мария,Москва,48,265072,0.03,False,2026-01-01 02:00:00
3,1004,Сергей,Москва,38,16515,0.58,False,2026-01-01 03:00:00
4,1005,Алексей,Казань,54,112206,0.17,False,2026-01-01 04:00:00


## Простая группировка — средний доход по флагу default

Хотим узнать: какой средний доход у хороших заёмщиков и какой — у тех, кто ушёл в дефолт?

In [ ]:
# Группируем по колонке default и считаем среднее для income
income_by_default = df.groupby('default')['income'].mean()
print("Средний доход по группам (default):")
print(income_by_default)
print(f"\nТип результата: {type(income_by_default)}")  # Series

Средний доход по группам (default):
default
False    257723.139906
True     255890.735294
Name: income, dtype: float64

Тип результата: <class 'pandas.Series'>


## Агрегация с несколькими функциями через agg()

Хотим для каждой группы (default) посчитать:
- Средний доход (mean)
- Минимальный и максимальный возраст
- Количество клиентов
- Стандартное отклонение DTI

In [ ]:
# передаём словарь {колонка: список_функций}
result = df.groupby('default').agg({
    'income': 'mean',
    'age': ['min', 'max'],
    'client_id': 'count',
    'dti': 'std'
})
print("Агрегация через словарь:")
print(result)
print(f"\nТип результата: {type(result)}")  # DataFrame

Агрегация через словарь:
                income age     client_id       dti
                  mean min max     count       std
default                                           
False    257723.139906  21  65      1694  0.287263
True     255890.735294  21  65       306  0.289889

Тип результата: <class 'pandas.DataFrame'>


In [ ]:
# через именованные агрегаты
result_named = df.groupby('default').agg(
    avg_income=('income', 'mean'),
    min_age=('age', 'min'),
    max_age=('age', 'max'),
    total_clients=('client_id', 'count'),
    std_dti=('dti', 'std')
)
print("Агрегация с именованными колонками:")
print(result_named)

Агрегация с именованными колонками:
            avg_income  min_age  max_age  total_clients   std_dti
default                                                          
False    257723.139906       21       65           1694  0.287263
True     255890.735294       21       65            306  0.289889


## Собственная функция в agg()

Допустим, нам нужен размах (range) — разница между максимумом и минимумом.
Такой функции нет в стандартном наборе, но ты можешь написать её сам.

In [ ]:
# своя функция для размаха
def data_range(x):
    """Возвращает разницу между максимумом и минимумом."""
    return x.max() - x.min()

# Применяем её к доходу
range_by_default = df.groupby('default')['income'].agg(data_range)
print("Размах дохода по группам (max - min):")
print(range_by_default)

# Можно сразу использовать лямбду
range_dti = df.groupby('default')['dti'].agg(lambda x: x.max() - x.min())
print("\nРазмах DTI по группам:")
print(range_dti)

Размах дохода по группам (max - min):
default
False    484609
True     483546
Name: income, dtype: int64

Размах DTI по группам:
default
False    1.0
True     1.0
Name: dti, dtype: float64


## Группировка по 2 колонкам: default + город

Хотим увидеть средний доход и количество клиентов в разрезе:
город — флаг дефолта.

In [ ]:
# Группируем по 2 колонкам
city_default = df.groupby(['city', 'default']).agg(
    avg_income=('income', 'mean'),
    count=('client_id', 'count')
)
print("Группировка по городу и default (иерархический индекс):")
print(city_default)
print(f"\nТип индекса: {type(city_default.index)}")  # MultiIndex

Группировка по городу и default (иерархический индекс):
                            avg_income  count
city            default                      
Екатеринбург    False    261984.545714    350
                True     254581.163265     49
Казань          False    258030.555891    331
                True     271461.250000     68
Москва          False    250758.829971    347
                True     271471.531250     64
Новосибирск     False    252471.395480    354
                True     251342.898551     69
Санкт-Петербург False    266320.852564    312
                True     225926.517857     56

Тип индекса: <class 'pandas.MultiIndex'>


In [ ]:
# Убираем иерархический индекс — делаем «плоскую» таблицу
city_default_flat = city_default.reset_index()
print("Та же группировка, но с плоским индексом:")
print(city_default_flat.head(10))

Та же группировка, но с плоским индексом:
              city  default     avg_income  count
0     Екатеринбург    False  261984.545714    350
1     Екатеринбург     True  254581.163265     49
2           Казань    False  258030.555891    331
3           Казань     True  271461.250000     68
4           Москва    False  250758.829971    347
5           Москва     True  271471.531250     64
6      Новосибирск    False  252471.395480    354
7      Новосибирск     True  251342.898551     69
8  Санкт-Петербург    False  266320.852564    312
9  Санкт-Петербург     True  225926.517857     56


## transform() — возвращаем агрегат обратно в таблицу

В отличие от agg(), transform() не уменьшает количество строк.
Он считает агрегат по группе и присваивает его каждой строке этой группы.
Это полезно, когда нужно добавить среднее по группе как новую колонку.

In [ ]:
# Добавляем колонку со средним доходом по городу
df['city_avg_income'] = df.groupby('city')['income'].transform('mean')

# Смотрим: для каждой строки свой городской средний доход
print("Данные с колонкой city_avg_income:")
print(df[['city', 'income', 'city_avg_income']].head(10))

moscow_rows = df[df['city'] == 'Москва'][['city', 'income', 'city_avg_income']]
print(f"\nМосква — средний доход: {moscow_rows['city_avg_income'].iloc[0]:.0f}")
print(f"Первые 5 строк Москвы:")
print(moscow_rows.head())

Данные с колонкой city_avg_income:
              city  income  city_avg_income
0  Санкт-Петербург   83968    260173.888587
1     Екатеринбург   75430    261075.358396
2           Москва  265072    253984.165450
3           Москва   16515    253984.165450
4           Казань  112206    260319.496241
5      Новосибирск  260282    252287.314421
6           Казань  250702    260319.496241
7      Новосибирск  422821    252287.314421
8  Санкт-Петербург  333384    260173.888587
9  Санкт-Петербург   20586    260173.888587

Москва — средний доход: 253984
Первые 5 строк Москвы:
      city  income  city_avg_income
2   Москва  265072     253984.16545
3   Москва   16515     253984.16545
13  Москва   46186     253984.16545
16  Москва  229159     253984.16545
17  Москва  400134     253984.16545


## сравнить профили хороших и плохих заёмщиков


In [ ]:
# Полный профиль заёмщика
profile = df.groupby('default').agg(
    total=('client_id', 'count'),
    avg_income=('income', 'mean'),
    median_income=('income', 'median'),
    avg_age=('age', 'mean'),
    avg_dti=('dti', 'mean'),
    min_age=('age', 'min'),
    max_age=('age', 'max')
)
print("Профиль заёмщиков (default vs no default):")
print(profile)

print(f"\nДоля дефолтов: {df['default'].mean():.1%}")

Профиль заёмщиков (default vs no default):
         total     avg_income  median_income    avg_age   avg_dti  min_age  \
default                                                                      
False     1694  257723.139906       257455.0  43.149351  0.496411       21   
True       306  255890.735294       257430.5  43.594771  0.477582       21   

         max_age  
default           
False         65  
True          65  

Доля дефолтов: 15.3%


## count, nunique, sum


In [ ]:
# Сгруппируем по городу и посчитаем:
city_stats = df.groupby('city').agg(
    total_clients=('client_id', 'count'),      # сколько всего клиентов
    unique_names=('name', 'nunique'),          # сколько уникальных имён
    total_income=('income', 'sum'),            # суммарный доход
    defaulter_count=('default', 'sum')         # сумма True — количество дефолтников
)
print("Статистика по городам:")
print(city_stats)

Статистика по городам:
                 total_clients  unique_names  total_income  defaulter_count
city                                                                       
Екатеринбург               399            10     104169068               49
Казань                     399            10     103867479               68
Москва                     411            10     104387492               64
Новосибирск                423            10     106717534               69
Санкт-Петербург            368            10      95743991               56
